# 02 — Region split

Turns the per-year distance table into a per-region split table and separates locations
into three groups:

| Group | Timestamps | Output |
|---|---|---|
| `train_test` | 3 | `region_split.parquet` — full train/test split |
| `inference_only` | 2 | `region_inference_only.parquet` — will get predictions, no train/test |
| `missing_data` | 1 | dropped — insufficient data for any prediction |

**Inputs:**
- `03_features/EXPERIMENT/dist_per_year.parquet`
- `wocu_post_processed_fase2_20260310.gpkg` — layer `summary_scope` (quality labels)

**Outputs** (to `03_features/EXPERIMENT/`):
- `region_split.parquet` — 3-timestamp OK regions with t1/t2/t3, v_train, v_test, split, is_nvo
- `region_inference_only.parquet` — 2-timestamp regions with t1/t2, v_last, last_dist

In [1]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (candidate / 'src').exists():
        _backend = candidate
        break
else:
    _backend = _cwd

os.chdir(_backend)
sys.path.insert(0, str(_backend))
print('cwd:', os.getcwd())

cwd: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend


In [2]:
import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import src.paths as PATHS

# ── Experiment config ─────────────────────────────────────────────────────────
EXPERIMENT   = '20260314'
RANDOM_SEED  = 42
TEST_SIZE    = 0.20
QUALITY_COL  = 'estimate_reliability_height_model'
CLUSTERS     = ['ijssel1', 'ijssel2', 'maas1', 'maas2', 'maas3', 'rijn', 'nederrijn']

FEATURES_DIR = PATHS.DATA_DIR / f'03_features/{EXPERIMENT}'
PROC_GPKG    = PATHS.DATA_DIR / '02_processed/erosion/wocu_post_processed_fase2_20260310.gpkg'

IN_PARQUET        = FEATURES_DIR / 'dist_per_year.parquet'
OUT_SPLIT         = FEATURES_DIR / 'region_split.parquet'
OUT_INFERENCE     = FEATURES_DIR / 'region_inference_only.parquet'

def get_cluster(loc_id: str) -> str:
    for c in CLUSTERS:
        if loc_id.startswith(c):
            return c
    return 'other'

print(f'Experiment : {EXPERIMENT}')
print(f'Input      : {IN_PARQUET}')

Experiment : 20260314
Input      : /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260314/dist_per_year.parquet


## 1. Load dist_per_year and quality labels

In [3]:
dist_per_year = pd.read_parquet(IN_PARQUET)
print(f'dist_per_year: {dist_per_year.shape}  ({dist_per_year["location_id"].nunique():,} locations)')

print('\nReading summary_scope quality labels ...')
scope_quality = gpd.read_file(PROC_GPKG, layer='summary_scope')
# Normalise location_id column name (may come as position_id)
if 'position_id' in scope_quality.columns and 'location_id' not in scope_quality.columns:
    scope_quality = scope_quality.rename(columns={'position_id': 'location_id'})
quality_map = scope_quality.set_index('location_id')[QUALITY_COL]
print(f'  {len(scope_quality):,} regions — quality breakdown:')
print(quality_map.value_counts().to_string())

dist_per_year: (30228, 5)  (10,484 locations)

Reading summary_scope quality labels ...


  16,772 regions — quality breakdown:
estimate_reliability_height_model
OK              8855
MISSING_DATA    5433
NOK             2484


## 2. Pivot to t1/t2/t3 per location and separate into timestamp groups

In [4]:
# Count timestamps per location
ts_counts = dist_per_year.groupby('location_id')['year'].count()
print('Timestamps per location (before quality filter):')
print(ts_counts.value_counts().sort_index().to_string())

locs_3ts  = ts_counts[ts_counts == 3].index
locs_2ts  = ts_counts[ts_counts == 2].index
locs_1ts  = ts_counts[ts_counts == 1].index
locs_4ts  = ts_counts[ts_counts >= 4].index

print(f'\n3-timestamp  : {len(locs_3ts):,}  → train/test candidates')
print(f'2-timestamp  : {len(locs_2ts):,}  → inference_only candidates')
print(f'1-timestamp  : {len(locs_1ts):,}  → MISSING_DATA (dropped)')
if len(locs_4ts):
    print(f'4+ timestamps: {len(locs_4ts):,}  → will use last 3 observations')

Timestamps per location (before quality filter):
year
1     142
2     940
3    9402

3-timestamp  : 9,402  → train/test candidates
2-timestamp  : 940  → inference_only candidates
1-timestamp  : 142  → MISSING_DATA (dropped)


In [5]:
def build_split_row(group, n_use=3):
    """Build a split record from the last n_use observations of a sorted group."""
    g = group.sort_values('year').tail(n_use).reset_index(drop=True)
    n = len(g)
    row = {'cluster': get_cluster(group.name), 'n_timestamps': n}
    for i, label in enumerate(['t1', 't2', 't3'][:n]):
        row[label]           = int(g.loc[i, 'year'])
        row[f'dist_{label}'] = g.loc[i, 'dist_m']
    if n >= 2:
        row['train_span_yr'] = row['t2'] - row['t1']
        row['v_train']       = (row['dist_t2'] - row['dist_t1']) / row['train_span_yr']
    if n == 3:
        row['test_span_yr']  = row['t3'] - row['t2']
        row['v_test']        = (row['dist_t3'] - row['dist_t2']) / row['test_span_yr']
    return pd.Series(row)

# Build for 3-timestamp locations (+ 4+ locations using last 3)
eligible_ids = ts_counts[ts_counts >= 3].index
split_records = (
    dist_per_year[dist_per_year['location_id'].isin(eligible_ids)]
    .groupby('location_id')
    .apply(build_split_row, include_groups=False)
)
print(f'Split records (≥3 ts): {split_records.shape}')

# Build for 2-timestamp locations
inference_records = (
    dist_per_year[dist_per_year['location_id'].isin(locs_2ts)]
    .groupby('location_id')
    .apply(build_split_row, include_groups=False)
)
print(f'Inference records (2 ts): {inference_records.shape}')

Split records (≥3 ts): (9402, 12)
Inference records (2 ts): (940, 8)


## 3. Apply quality filter (keep OK only)

In [6]:
split_records['quality'] = split_records.index.map(quality_map)

before = len(split_records)
features_ok = split_records[split_records['quality'] == 'OK'].copy()
after  = len(features_ok)

removed_nok     = (split_records['quality'] == 'NOK').sum()
removed_missing = (split_records['quality'] == 'MISSING_DATA').sum()
removed_unknown = split_records['quality'].isna().sum()

print(f'Before quality filter : {before:,}')
print(f'  Removed NOK         : {removed_nok:,}')
print(f'  Removed MISSING_DATA: {removed_missing:,}')
print(f'  Removed unknown/NaN : {removed_unknown:,}')
print(f'After quality filter  : {after:,}  ({after/before*100:.1f}% retained)')

# Also apply quality filter to inference_only
inference_records['quality'] = inference_records.index.map(quality_map)
inference_ok = inference_records[inference_records['quality'] == 'OK'].copy()
print(f'\nInference_only after quality filter: {len(inference_ok):,}')

Before quality filter : 9,402
  Removed NOK         : 1,958
  Removed MISSING_DATA: 0
  Removed unknown/NaN : 0
After quality filter  : 7,444  (79.2% retained)

Inference_only after quality filter: 650


## 4. Join is_nvo from VVR layer

In [7]:
print('Reading vvr_rates_of_change for is_nvo flag ...')
vvr_polys = gpd.read_file(PROC_GPKG, layer='vvr_rates_of_change')
scope_geom = gpd.read_file(PROC_GPKG, layer='summary_scope')
if 'position_id' in scope_geom.columns and 'location_id' not in scope_geom.columns:
    scope_geom = scope_geom.rename(columns={'position_id': 'location_id'})

# Scope regions that intersect any VVR polygon → is_nvo = True
scope_ok = scope_geom[scope_geom['location_id'].isin(features_ok.index)][['location_id', 'geometry']]
scope_ok = scope_ok.to_crs(vvr_polys.crs)
nvo_ids = set(
    gpd.sjoin(scope_ok, vvr_polys[['geometry']], how='inner', predicate='intersects')['location_id']
)
features_ok['is_nvo'] = features_ok.index.isin(nvo_ids)
inference_ok['is_nvo'] = inference_ok.index.isin(nvo_ids)

n_nvo = features_ok['is_nvo'].sum()
print(f'NVO regions (train/test): {n_nvo:,}  ({n_nvo/len(features_ok)*100:.1f}%)')
print(f'NVO regions (inference) : {inference_ok["is_nvo"].sum():,}')

Reading vvr_rates_of_change for is_nvo flag ...


NVO regions (train/test): 1,985  (26.7%)
NVO regions (inference) : 0


## 5. Stratified 80/20 train/test split

In [8]:
train_idx, test_idx = train_test_split(
    features_ok.index,
    test_size    = TEST_SIZE,
    random_state = RANDOM_SEED,
    stratify     = features_ok['cluster'],
)
features_ok['split'] = 'train'
features_ok.loc[test_idx, 'split'] = 'test'

split_counts = features_ok.groupby(['cluster', 'split']).size().unstack(fill_value=0)
split_counts['total'] = split_counts.sum(axis=1)
print('Train/test split per cluster:')
print(split_counts.to_string())
print(f'\nTotal: {len(features_ok):,}  (train={len(train_idx):,}, test={len(test_idx):,})')

Train/test split per cluster:
split      test  train  total
cluster                      
ijssel1     362   1448   1810
ijssel2      57    230    287
maas1        25    102    127
maas2        36    142    178
maas3       536   2145   2681
nederrijn   240    958   1198
rijn        233    930   1163

Total: 7,444  (train=5,955, test=1,489)


## 6. Sanity checks

In [9]:
# No nulls in key columns
key_cols = ['t1', 't2', 't3', 'dist_t1', 'dist_t2', 'dist_t3', 'v_train', 'v_test', 'train_span_yr', 'test_span_yr']
null_counts = features_ok[key_cols].isnull().sum()
assert null_counts.sum() == 0, f'Unexpected nulls:\n{null_counts[null_counts > 0]}'

# Year ordering: t1 < t2 < t3
assert (features_ok['t1'] < features_ok['t2']).all(), 't1 >= t2 for some rows'
assert (features_ok['t2'] < features_ok['t3']).all(), 't2 >= t3 for some rows'

# No overlap between train_test and inference_only
overlap = set(features_ok.index) & set(inference_ok.index)
assert len(overlap) == 0, f'{len(overlap)} locations in both train/test and inference_only'

print('All checks passed.')
print(f'region_split        : {features_ok.shape}')
print(f'region_inference_only: {inference_ok.shape}')

All checks passed.
region_split        : (7444, 15)
region_inference_only: (650, 10)


## 6b. Erosion volume from `erosion_vlakken_filtered`

For each scope region, match erosion polygons to:
- **Train period**: `year_before == t1` AND `year_after == t2`
- **Test period** : `year_before == t2` AND `year_after == t3`

Volume summed per region; regions with no matching polygon → 0.
Annual rate = total volume / span_years. Alias `erosion_vol_rate_t1 = erosion_vol_train_rate`.

In [10]:
print('Loading erosion_vlakken_filtered ...')
ev_raw = gpd.read_file(PROC_GPKG, layer='erosion_vlakken_filtered')

ev = (ev_raw[['location_id', 'year_before', 'year_after', 'area', 'erosion_volume']]
      .drop_duplicates()
      .reset_index(drop=True))
print(f'  Raw rows: {len(ev_raw):,}  →  after dedup: {len(ev):,}')
print(f'  Unique location_ids: {ev.location_id.nunique():,}')

t_lookup = features_ok[['t1', 't2', 't3']].copy()
ev_joined = ev.merge(t_lookup, left_on='location_id', right_index=True, how='inner')

train_vol = (ev_joined[
    (ev_joined['year_before'] == ev_joined['t1']) &
    (ev_joined['year_after']  == ev_joined['t2'])
].groupby('location_id')['erosion_volume'].sum().rename('erosion_vol_train_m3'))

test_vol = (ev_joined[
    (ev_joined['year_before'] == ev_joined['t2']) &
    (ev_joined['year_after']  == ev_joined['t3'])
].groupby('location_id')['erosion_volume'].sum().rename('erosion_vol_test_m3'))

features_ok = features_ok.join(train_vol).join(test_vol)
features_ok['erosion_vol_train_m3'] = features_ok['erosion_vol_train_m3'].fillna(0)
features_ok['erosion_vol_test_m3']  = features_ok['erosion_vol_test_m3'].fillna(0)

features_ok['erosion_vol_train_rate'] = features_ok['erosion_vol_train_m3'] / features_ok['train_span_yr']
features_ok['erosion_vol_test_rate']  = features_ok['erosion_vol_test_m3']  / features_ok['test_span_yr']
features_ok['erosion_vol_rate_t1']    = features_ok['erosion_vol_train_rate']

n_with = (features_ok['erosion_vol_train_m3'] > 0).sum()
print(f'\nRegions with measured erosion (train): {n_with:,} / {len(features_ok):,}  ({n_with/len(features_ok)*100:.1f}%)')
print(f'erosion_vol_rate_t1: mean={features_ok["erosion_vol_rate_t1"].mean():.2f}  '
      f'p75={features_ok["erosion_vol_rate_t1"].quantile(0.75):.2f}  '
      f'max={features_ok["erosion_vol_rate_t1"].max():.2f}  m³/yr')

Loading erosion_vlakken_filtered ...


  Raw rows: 25,194  →  after dedup: 24,024
  Unique location_ids: 5,267

Regions with measured erosion (train): 2,007 / 7,444  (27.0%)
erosion_vol_rate_t1: mean=7.28  p75=0.37  max=1934.22  m³/yr


## 7. Save

In [11]:
# region_split — train/test columns
split_cols = [
    'cluster', 'n_timestamps',
    't1', 't2', 't3',
    'train_span_yr', 'test_span_yr',
    'dist_t1', 'dist_t2', 'dist_t3',
    'v_train', 'v_test',
    'is_nvo', 'quality', 'split',
    'erosion_vol_train_rate', 'erosion_vol_test_rate', 'erosion_vol_rate_t1',
]
features_ok[split_cols].to_parquet(OUT_SPLIT)
print(f'Saved region_split         → {OUT_SPLIT}  {features_ok.shape}')

# region_inference_only — minimal columns needed for downstream feature engineering
inf_cols = [
    'cluster', 'n_timestamps',
    't1', 't2',
    'train_span_yr',
    'dist_t1', 'dist_t2',
    'v_train',
    'is_nvo', 'quality',
]
# For inference_only the 'last' observation is t2
inference_out = inference_ok[inf_cols].copy()
inference_out.to_parquet(OUT_INFERENCE)
print(f'Saved region_inference_only → {OUT_INFERENCE}  {inference_out.shape}')

Saved region_split         → /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260314/region_split.parquet  (7444, 20)


Saved region_inference_only → /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260314/region_inference_only.parquet  (650, 10)
